In [ ]:
# [COLAB SETUP]
import sys
import os

if "google.colab" in sys.modules:
    print("Running in Google Colab. Setting up environment...")
    
    from google.colab import drive
    drive.mount('/content/drive')
    
    repo_path = '/content/drive/MyDrive/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit'
    
    if not os.path.exists(repo_path):
        print(f"Cloning repository into {repo_path}...")
        os.makedirs('/content/drive/MyDrive', exist_ok=True)
        os.system(f'git clone https://github.com/Maleesha-K/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit.git {repo_path}')
        
    os.chdir(repo_path + '/data_pipeline')
    print("Installing dependencies...")
    os.system('pip install -q pandas datasets')
    print("Setup complete!")


In [ ]:
import os
import json
import pandas as pd
from datasets import load_dataset

# Auto-resolve the project root if running manually
if not os.path.exists("Makefile") and os.path.exists("../../Makefile"):
    os.chdir("../../")

output_dir = 'datasets/finetuning'
train_jsonl_path = os.path.join(output_dir, "train.jsonl")
output_mixed_path = os.path.join(output_dir, "train_mixed.jsonl")

# Target languages to preserve (excluding Sinhala, Pali, Sanskrit)
TARGET_LANGUAGES = {
    "eng": "en",  # English
    "hin": "hi",  # Hindi
    "arb": "ar",  # Arabic
    "fra": "fr",  # French
    "deu": "de",  # German
    "ben": "bn",  # Bengali
    "tam": "ta",  # Tamil
}

SAMPLES_PER_LANG = 5000


In [ ]:
print("Loading Aya Dataset from HuggingFace...")
aya_dataset = load_dataset("CohereLabs/aya_dataset", split="train")

print("Filtering for target rehearsal languages...")
rehearsal_records = []
lang_counts = {lang: 0 for lang in TARGET_LANGUAGES.keys()}

for row in aya_dataset:
    lang = row['language_code']
    if lang in TARGET_LANGUAGES and lang_counts[lang] < SAMPLES_PER_LANG:
        rehearsal_records.append({
            "text": row["inputs"],
            "label": lang,
            "source": "aya_dataset"
        })
        lang_counts[lang] += 1

    # Stop early if all quotas are met
    if all(count == SAMPLES_PER_LANG for count in lang_counts.values()):
        break

print(f"Sampled rehearsal distributions from Aya Dataset:")
for lang, count in lang_counts.items():
    print(f"  {lang}: {count}")


In [ ]:
print(f"\nLoading original training data from {train_jsonl_path}...")
train_records = []
if os.path.exists(train_jsonl_path):
    with open(train_jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            train_records.append(json.loads(line))
    print(f"Loaded {len(train_records)} original training records (Sinhala, Pali, Sanskrit).")
else:
    print(f"WARNING: {train_jsonl_path} not found. Please run setup_finetune_data.ipynb first.")

print("\nCombining and shuffling datasets...")
import random
mixed_records = train_records + rehearsal_records
random.shuffle(mixed_records)

with open(output_mixed_path, 'w', encoding='utf-8') as f:
    for record in mixed_records:
        f.write(json.dumps(record, ensure_ascii=False) + '\n')

print(f"Created {output_mixed_path} with {len(mixed_records)} total mixed records.")
